# Hour 3 · Letters as social networks

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alexsosn/ugarit-dh-workshop/blob/main/notebooks/3b_letter_networks.ipynb) [![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/alexsosn/ugarit-dh-workshop/main?labpath=notebooks%2F3b_letter_networks.ipynb)


**Goal:** turn the corpus of **letters** into a graph of correspondents.

**Needs:** `networkx`. Ugaritic letters open with a fixed address formula: **`tḥm X`** = 'message of X' (sender) and **`l Y rgm`** = 'to Y, say' (recipient).

*Reading:* `docs/06-letters-networks.md`.

## 0. Setup


In [ ]:
# === SETUP — run me first (works in Google Colab, Binder, and locally) ===
import os, sys, subprocess

if "google.colab" in sys.modules:                      # we're on Colab
    REPO_URL = "https://github.com/alexsosn/ugarit-dh-workshop.git"
    REPO_DIR = "/content/ugarit-dh-workshop"
    if not os.path.isdir(REPO_DIR):                    # clone the repo once
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(os.path.join(REPO_DIR, "notebooks"))      # work from notebooks/
    # Colab already ships numpy/pandas/scikit-learn/matplotlib/plotly/networkx;
    # only umap-learn is usually missing. Install just that (fast).
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "umap-learn"], check=False)

# make data/loader.py importable (we run from the notebooks/ folder)
sys.path.insert(0, os.path.abspath(".."))
from data.loader import load_texts

texts = load_texts()     # 1st Colab run downloads the CUC JSONL from HuggingFace, then caches it
print(f"Loaded {len(texts)} tablets — genre of the first one: {texts[0]['genre']}")

## 1. Parse (sender → recipient) from the address formula

In [ ]:
# particles that are not names (drop them if the formula parse grabs one)
STOP = {"l", "w", "b", "k", "ʿm", "il", "ilm"}

def parse_letter(t):
    toks = t["tokens"]
    sender = recipient = None
    for i, w in enumerate(toks[:-1]):
        if w == "tḥm" and sender is None:
            sender = toks[i+1]
        if w == "l" and recipient is None:
            recipient = toks[i+1]
    return sender, recipient

edges = []
for t in texts:
    if t["genre"] != "letter": continue
    s, r = parse_letter(t)
    if s and r and s != r and s not in STOP and r not in STOP:
        edges.append((s, r, t["ktu"]))
print(len(edges), "edges parsed")
edges[:12]

## 2. Build the directed correspondence graph

In [ ]:
import networkx as nx
G = nx.DiGraph()
for s, r, k in edges:
    G.add_edge(s, r, ktu=k)
print("people:", G.number_of_nodes(), " letters:", G.number_of_edges())

## 3. Draw it

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(11,8))
pos = nx.spring_layout(G, seed=1, k=0.6)
sizes = [300 + 400*G.degree(n) for n in G.nodes()]
nx.draw(G, pos, with_labels=True, node_size=sizes, node_color="#cfe8ff",
        font_size=8, arrows=True, edge_color="#999")
plt.title("Ugaritic correspondence network (CUC letters)"); plt.show()

## 4. Who is central?

In [ ]:
import pandas as pd
pd.Series(nx.degree_centrality(G)).sort_values(ascending=False).head(10)

## 5. Discussion
- Central nodes are often titles/relations (*mlk* king, *umy* 'my mother/lady') rather than unique persons — a real interpretive caveat.
- **Fragmentary preservation** biases the graph; a missing edge ≠ no relationship.

> **Production version:** UgaritLab `build_name_graph.py` builds a co-occurrence graph of named entities (PN/DN/TN/RN) from DULAT matched in UDB tablets — far richer than this parser.